<a href="https://colab.research.google.com/github/emmanguyen01-spec/Coding-Exercise---ML-Basics/blob/main/ML/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import os
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data Source:
# Kaggle - USA Real Estate Dataset by Ahmed Shahriar Sakib
# https://www.kaggle.com/datasets/ahmedshahriarsakib/usa-real-estate-dataset

# Download latest version
path = kagglehub.dataset_download(
    "ahmedshahriarsakib/usa-real-estate-dataset"
)

print("Path to dataset files:", path)

# See which files are inside the downloaded dataset
print(os.listdir(path))

df = pd.read_csv(
    os.path.join(path, "realtor-data.zip.csv")
)

# Preview dataset
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

# Keep the columns needed for the assignment
df = df[['price', 'house_size']]

# Remove missing values
df = df.dropna()

# Rename house_size to match the starter code
df = df.rename(columns={'house_size': 'square_footage'})

# Use 1000 records
df = df.sample(n=1000, random_state=42).reset_index(drop=True)

# Add the required location column
np.random.seed(42)
df['location'] = np.random.choice(
    ['Downtown', 'Suburb', 'Rural'],
    size=len(df)
)

# Preview the data
print(df.head())

# Features and target
X = df[['square_footage', 'location']]
y = df['price']

# Preprocessing: One-hot encode the location column
preprocessor = ColumnTransformer(
transformers=[
('location', OneHotEncoder(sparse_output=False), ['location'])
], remainder='passthrough')

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
('preprocessor', preprocessor),
('regressor', LinearRegression())
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

# Train model
model.fit(X_train, y_train)

# Make prediction for a new house: 2000 sq ft in Downtown
new_house = pd.DataFrame({
    'square_footage': [2000],
    'location': ['Downtown']
})

predicted_price = model.predict(new_house)

print(
    f"\nPredicted price for a 2000 sq ft house in Downtown: "
    f"${predicted_price[0]:,.2f}"
)

# Display model coefficients
feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['location']
    .get_feature_names_out(['location'])
).tolist() + ['square_footage']

coefficients = model.named_steps['regressor'].coef_

print("\nModel Coefficients:")

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Using Colab cache for faster access to the 'usa-real-estate-dataset' dataset.
Path to dataset files: /kaggle/input/usa-real-estate-dataset
['realtor-data.zip.csv']
   brokered_by    status     price  bed  bath  acre_lot     street  \
0     103378.0  for_sale  105000.0  3.0   2.0      0.12  1962661.0   
1      52707.0  for_sale   80000.0  4.0   2.0      0.08  1902874.0   
2     103379.0  for_sale   67000.0  2.0   1.0      0.15  1404990.0   
3      31239.0  for_sale  145000.0  4.0   2.0      0.10  1947675.0   
4      34632.0  for_sale   65000.0  6.0   2.0      0.05   331151.0   

         city        state  zip_code  house_size prev_sold_date  
0    Adjuntas  Puerto Rico     601.0       920.0            NaN  
1    Adjuntas  Puerto Rico     601.0      1527.0            NaN  
2  Juana Diaz  Puerto Rico     795.0       748.0            NaN  
3       Ponce  Puerto Rico     731.0      1800.0            NaN  
4    Mayaguez  Puerto Rico     680.0         NaN            NaN  

Dataset shape:
(22